# 并查集

并查集（Disjoint Set Union, DSU），也常被称为 Union-Find，是一种用于维护若干个互不相交集合的数据结构。

它主要支持两类操作：
- `find(x)`: 查询元素 `x` 所在集合的代表元素；
- `union(x, y)`: 合并元素 `x` 与元素 `y` 所在的两个集合。

如果两个元素的代表元素相同，就说明它们属于同一个集合。因此，并查集特别适合处理“连通性”问题，例如判断两个点是否连通、统计连通分量个数、Kruskal 最小生成树、朋友圈/社交关系分组等问题。

## 基本思想

并查集通常把每个集合组织成一棵树。树中的每个节点表示一个元素，每个节点保存一个父节点指针 `parent`。

对于一个集合，我们选取树根作为这个集合的代表元素。树根的父节点是它自己，即：

```python
parent[root] == root
```

例如，假设有 5 个元素，初始时每个元素单独构成一个集合：

```text
{0}, {1}, {2}, {3}, {4}
```

此时每个元素都是自己的代表元素。若执行 `union(0, 1)` 和 `union(1, 2)`，那么 `0, 1, 2` 会被合并到同一个集合中：

```text
{0, 1, 2}, {3}, {4}
```

这时 `find(0)`、`find(1)`、`find(2)` 应该返回同一个代表元素。代表元素具体是谁并不重要，重要的是同一集合中的所有元素经过 `find` 后结果一致。

## 两个核心优化

如果直接用树来表示集合，树可能退化成一条链，此时 `find` 的时间复杂度可能达到 $O(N)$。为了避免这种情况，实际实现中通常使用两个优化。

### 1. 路径压缩

在执行 `find(x)` 时，如果一路向上找到了根节点，就顺手把路径上的节点直接挂到根节点下面。

这样以后再查询这些节点时，就可以更快地找到代表元素。

### 2. 按大小或按秩合并

在执行 `union(x, y)` 时，不随意把一棵树接到另一棵树下面，而是把较小的树接到较大的树下面。这样可以尽量避免树变高。

常见做法有两种：
- 按大小合并（union by size）：记录每棵树包含多少个节点；
- 按秩合并（union by rank）：记录树高的近似上界。

下面的实现使用按大小合并。

## 时间复杂度

同时使用路径压缩和按大小/按秩合并后，并查集的单次操作均摊时间复杂度为：

$$
O(\alpha(N))
$$

其中 $\alpha(N)$ 是反阿克曼函数。它增长极其缓慢，在实际数据规模下几乎可以看作常数。因此，我们通常认为并查集的 `find` 和 `union` 操作接近 $O(1)$。

空间复杂度为 $O(N)$，主要用于保存每个元素的父节点和集合大小。

In [ ]:
class DisjointSetUnion:
    """并查集，支持路径压缩和按集合大小合并。"""

    def __init__(self, n: int):
        if n < 0:
            raise ValueError("n must be non-negative")

        self.parent = list(range(n))
        self.size = [1] * n
        self.count = n  # 当前连通分量/集合数量

    def __len__(self) -> int:
        return len(self.parent)

    def _validate(self, x: int) -> None:
        if not 0 <= x < len(self.parent):
            raise IndexError(f"index {x} is out of range")

    def find(self, x: int) -> int:
        """返回 x 所在集合的代表元素。"""
        self._validate(x)

        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # 路径压缩
        return self.parent[x]

    def union(self, x: int, y: int) -> bool:
        """
        合并 x 和 y 所在的集合。

        如果原本不在同一个集合，合并后返回 True；
        如果原本已经在同一个集合，返回 False。
        """
        root_x = self.find(x)
        root_y = self.find(y)

        if root_x == root_y:
            return False

        # 按大小合并：让小集合的根指向大集合的根
        if self.size[root_x] < self.size[root_y]:
            root_x, root_y = root_y, root_x

        self.parent[root_y] = root_x
        self.size[root_x] += self.size[root_y]
        self.count -= 1
        return True

    def connected(self, x: int, y: int) -> bool:
        """判断 x 和 y 是否属于同一个集合。"""
        return self.find(x) == self.find(y)

    def component_size(self, x: int) -> int:
        """返回 x 所在集合的元素个数。"""
        return self.size[self.find(x)]

    def components(self) -> dict[int, list[int]]:
        """返回当前所有集合，键为代表元素，值为该集合中的元素列表。"""
        groups: dict[int, list[int]] = {}
        for x in range(len(self.parent)):
            root = self.find(x)
            groups.setdefault(root, []).append(x)
        return groups


## 使用示例

假设有编号为 `0` 到 `6` 的 7 个元素。下面通过若干次合并操作来维护它们之间的连通关系。

In [ ]:
dsu = DisjointSetUnion(7)

dsu.union(0, 1)
dsu.union(1, 2)
dsu.union(3, 4)
dsu.union(5, 6)

print(dsu.connected(0, 2))  # True
print(dsu.connected(0, 4))  # False
print(dsu.component_size(1))  # 3
print(dsu.count)  # 3
print(dsu.components())


继续执行 `union(2, 4)`，会把 `{0, 1, 2}` 和 `{3, 4}` 合并成一个更大的集合。

In [ ]:
dsu.union(2, 4)

print(dsu.connected(0, 4))  # True
print(dsu.component_size(3))  # 5
print(dsu.count)  # 2
print(dsu.components())


## 典型应用：统计无向图的连通分量

给定一个无向图，若图中有 `n` 个顶点和若干条边，可以把每条边 `(u, v)` 看作一次合并操作 `union(u, v)`。所有边处理完后，并查集中的集合数量就是图的连通分量数量。

In [ ]:
def count_components(n: int, edges: list[tuple[int, int]]) -> int:
    dsu = DisjointSetUnion(n)
    for u, v in edges:
        dsu.union(u, v)
    return dsu.count


edges = [(0, 1), (1, 2), (3, 4), (5, 6)]
print(count_components(7, edges))  # 3


## 小结

并查集适合解决动态合并集合、判断元素是否属于同一集合的问题。

它的关键点是：
- 用树表示集合，树根作为集合代表；
- `find` 用于查找代表元素；
- `union` 用于合并两个集合；
- 路径压缩可以降低后续查询成本；
- 按大小/按秩合并可以避免树退化；
- 优化后的并查集在实践中几乎可以看作常数时间。